# 06. Capacity Stress Analysis
Identify relative high-load observations and sustained positive system net flow. These are analytical pressure indicators, not official capacity thresholds.

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_raw_data
from src.preprocessing import clean_data
from src.feature_engineering import add_features

raw = load_raw_data()
clean = clean_data(raw)
features = add_features(clean)

In [3]:
threshold = features.total_system_load.quantile(.90)
print('Full-sample 90th percentile:', threshold)
print('High-load observations:', int(features.high_system_load_flag.sum()))

Full-sample 90th percentile: 9920.9
High-load observations: 72


In [4]:
stress = features.loc[features.high_system_load_flag, ['date','cbp_custody','hhs_care','total_system_load','system_net_flow']]
display(stress.sort_values('total_system_load', ascending=False).head(20))

,date,cbp_custody,hhs_care,total_system_load,system_net_flow
224,2023-12-20,246.0,11516,11762.0,-254.0
225,2023-12-21,228.0,11375,11603.0,-369.0
223,2023-12-19,185.0,11291,11476.0,-167.0
220,2023-12-14,157.0,11242,11399.0,-246.0
161,2023-09-19,287.0,11095,11382.0,-127.0
162,2023-09-20,310.0,11064,11374.0,-177.0
219,2023-12-13,181.0,11192,11373.0,-219.0
163,2023-09-21,285.0,11067,11352.0,-260.0
226,2023-12-25,196.0,11144,11340.0,-257.0
229,2023-12-28,107.0,11230,11337.0,-289.0


In [5]:
from src.validation import stock_flow_reconciliation
reconciliation = stock_flow_reconciliation(clean)
reconciliation[['cbp_reconciliation_error','hhs_reconciliation_error']].describe()

,cbp_reconciliation_error,hhs_reconciliation_error
count,558.0,558.0
mean,34.333333,67.71147
std,40.78529,107.5113
min,-102.0,-170.0
25%,5.0,1.0
50%,27.5,9.0
75%,57.0,131.0
max,334.0,541.0
